In [1]:
import pandas as pd
import torch as pt
import torch.nn as nn
from time import time
import pyarrow.parquet as pq
import random
import itertools as it
import gc
import tqdm
import numpy as np
from ucf_atd_model.data import data_loc
from ucf_atd_model.datasets.create_link_data import calculate_link_features, haversine_distance_m
from ucf_atd_model.datasets.create_20class_data import subset, setidx, boolfilter, paddata, const_data
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, accuracy_score
from ucf_atd_model.datasets.create_link_data import calculate_link_features, haversine_distance_m, project_forward
import matplotlib.pyplot as plt
from datetime import datetime
import scipy as sp
from ucf_atd_model.c20_consts import *

In [3]:
device = pt.device("cpu")

In [26]:
badnames = [x for x in full_names if x.endswith("_16")]

ynames = [x for x in ynames if not x.endswith("_16")]
xnames = [x for x in colnames if x not in badnames]

# Setup the model
inp_dim = len(colnames) - len(badnames) + 1
h_dim = 2000
out_dim = n_norm_classes + 1 - 1

model = nn.Sequential(
    nn.Linear(inp_dim, h_dim),
    nn.ReLU(),
    nn.Linear(h_dim, h_dim),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(h_dim, h_dim // 2),
    nn.ReLU(),
    nn.Dropout(0.5),
    # nn.Linear(h_dim // 2, h_dim // 2),
    # nn.ReLU(),
    # nn.Dropout(0.2),
    nn.Linear(h_dim // 2, out_dim),
).to(device)

model.load_state_dict(pt.load("checkpoints/epoch_180.pt", weights_only=True, map_location=device))
model.eval()

Sequential(
  (0): Linear(in_features=231, out_features=2000, bias=True)
  (1): ReLU()
  (2): Linear(in_features=2000, out_features=2000, bias=True)
  (3): ReLU()
  (4): Dropout(p=0.1, inplace=False)
  (5): Linear(in_features=2000, out_features=1000, bias=True)
  (6): ReLU()
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=1000, out_features=17, bias=True)
)

In [27]:
xmean = pt.load("xmean.pt").float()
xstd = pt.load("xstd.pt").float()

In [28]:
# Important dataset features
num_loops = None
validation = [0, 1, 2]

test_X = None
test_y = None

table: pd.DataFrame = None
with pq.ParquetFile(data_loc("c20_data/class20_0.parquet")) as fulldata:
    testData = fulldata.read_row_groups(validation).to_pandas()
    
    test_X = pt.from_numpy(testData.drop(ynames + badnames, axis=1).to_numpy()).float()
    test_y = pt.from_numpy(testData[ynames].to_numpy()).float()
    
    y_test_mask = pt.zeros_like(test_y, dtype=pt.float32)
    y_test_mask[:, -1] = 0
    y_test_mask[:, 0] = 0

    num_ft_sets = n_norm_classes - 1
    for i in range(1, num_ft_sets):
        ft_names = getNormFeatures(i)
        feats = testData[ft_names]
        y_test_mask[:, i] = pt.from_numpy(((feats == -1).all(axis=1) * -1e8).to_numpy()).float()

    test_X = (test_X - xmean) / xstd

In [29]:
y_test_mask

tensor([[         0., -100000000., -100000000.,  ..., -100000000.,
         -100000000.,          0.],
        [         0., -100000000., -100000000.,  ..., -100000000.,
         -100000000.,          0.],
        [         0., -100000000., -100000000.,  ..., -100000000.,
         -100000000.,          0.],
        ...,
        [         0.,         -0.,         -0.,  ..., -100000000.,
         -100000000.,          0.],
        [         0.,         -0.,         -0.,  ..., -100000000.,
         -100000000.,          0.],
        [         0.,         -0.,         -0.,  ..., -100000000.,
         -100000000.,          0.]])

In [30]:
model.eval()
preds = None
true_y = None
with pt.no_grad():
    preds: pt.Tensor = model(test_X)
    preds = preds.cpu().detach().numpy()
    true_y = test_y.cpu().detach().numpy()

preds_masked = preds + y_test_mask.detach().numpy()

In [31]:
simple_preds = np.zeros_like(preds)
simple_preds[np.arange(simple_preds.shape[0]), np.argmax(preds, axis=1)] = 1

In [32]:
simple_preds_masked = np.zeros_like(preds)
simple_preds_masked[np.arange(simple_preds_masked.shape[0]), np.argmax(preds_masked, axis=1)] = 1

In [33]:
print(classification_report(true_y, simple_preds_masked))

              precision    recall  f1-score   support

           0       0.94      0.84      0.89     35050
           1       0.90      0.85      0.87     28515
           2       0.87      0.87      0.87     25098
           3       0.84      0.86      0.85     21969
           4       0.85      0.83      0.84     19873
           5       0.87      0.78      0.82     17458
           6       0.89      0.73      0.80     15753
           7       0.90      0.70      0.79     14057
           8       0.88      0.68      0.77     13112
           9       0.87      0.66      0.75     12211
          10       0.79      0.63      0.70     11223
          11       0.71      0.58      0.64     10388
          12       0.65      0.55      0.59      9857
          13       0.61      0.52      0.56      9638
          14       0.51      0.55      0.53      9187
          15       0.47      0.62      0.54      6879
          16       0.67      0.67      0.67     27109

   micro avg       0.81   

In [16]:
overall_roc = roc_auc_score(true_y, sp.special.softmax(preds, axis=1), multi_class="ovr")

In [17]:
overall_roc2 = roc_auc_score(true_y, sp.special.softmax(preds_masked, axis=1), multi_class="ovr")

In [18]:
accuracy_score(true_y, simple_preds_masked)

0.7451627947538646

In [34]:
def run_model(df, model):
    """Implements the final ML-Enhanced Tracking algorithm."""
    df = df.sort_values('time').reset_index(drop=True)
    df['track_id'] = -1

    next_track_id = 0
    
    n = df.shape[0]
    lastPtInTrack = {
        "time": np.repeat(pd.Timestamp(year=1970, month=1, day=1, hour=0, minute=0, second=0).to_numpy(), n), 
        "lat": np.repeat(-1.0, n), 
        "lon": np.repeat(-1.0, n), 
        "speed": np.repeat(-1.0, n), 
        "course": np.repeat(-1.0, n),
        "track_id_true": np.repeat(-1, n)
    }

    for i in tqdm.tqdm(range(len(df))):
        p_current = df.iloc[i]

        if next_track_id == 0:
            df.loc[i, 'track_id'] = next_track_id
            setidx(lastPtInTrack, i, p_current)
            next_track_id += 1
            continue


        active_tracks_df = subset(lastPtInTrack, next_track_id)
        
        time_diff = (p_current["time"].to_numpy() - active_tracks_df["time"]).astype("timedelta64[s]").astype("int")

        max_dist_m = time_diff * 30 * 0.5144
        real_dist = haversine_distance_m(active_tracks_df["lat"], active_tracks_df["lon"], p_current["lat"], p_current["lon"])
        
        kinematic_errors = haversine_distance_m(p_current["lat"], p_current["lon"], *project_forward(active_tracks_df['lat'], active_tracks_df['lon'], active_tracks_df['speed'], active_tracks_df['course'], time_diff))
        error_cutoff = np.sort(kinematic_errors)[:n_norm_classes].max()
        kinematic_filter = kinematic_errors < error_cutoff
        
        loc_filter = real_dist < max_dist_m
        timeCorrect: np.ndarray = (0 < time_diff)
        big_filter = loc_filter & timeCorrect & kinematic_filter
        idxs = np.arange(len(timeCorrect))

        # Create data if we find data points within the filters
        if np.any(big_filter):
            all_data = paddata(calculate_link_features(boolfilter(active_tracks_df, big_filter), p_current, eval=True))
            maindata = all_data[normal_features][:-1].to_numpy()
            otherdata = all_data[currpt_features].iloc[0].to_numpy()
            
            # Predict on this data
            toappend = np.zeros((n_norm_classes - 1) * len(normal_features) + len(currpt_features))
            raveled = np.ravel(maindata)
            toappend[:raveled.shape[0]] = raveled
            toappend[raveled.shape[0]:] = otherdata

            tensorIn = ((pt.from_numpy(toappend).float() - xmean) / xstd).to(device)

            modelOut = model(tensorIn).detach().cpu().numpy()
            outMask = np.zeros_like(modelOut)
            outMask[:-1] = np.all(maindata == -1, axis=1) * -1e8
            outMask[-1] = 0
            modelOut = modelOut + outMask
            
            argMaxModelOut = np.argmax(modelOut)

            # Assignment with a confidence threshold
            if argMaxModelOut != len(modelOut) - 1:
                best_match_track_id = None
                try:
                    best_match_track_id = idxs[big_filter][argMaxModelOut]
                except IndexError:
                    print("Bad")
                    best_match_track_id = next_track_id
                    next_track_id += 1

                df.loc[i, 'track_id'] = best_match_track_id
                setidx(lastPtInTrack, best_match_track_id, p_current)
            else:
                df.loc[i, 'track_id'] = next_track_id
                setidx(lastPtInTrack, next_track_id, p_current)
                next_track_id += 1
        else:
            df.loc[i, 'track_id'] = next_track_id
            setidx(lastPtInTrack, next_track_id, p_current)
            next_track_id += 1

    return df[['point_id', 'track_id']]

truth_df = pd.read_csv(data_loc("dataset1_truth.csv"))

truth_df["time"] = pd.to_datetime(truth_df["time"])
truth_df["time"] = truth_df["time"].apply(lambda x: datetime.combine(datetime(1970, 1, 1, 0, 0, 0).date(), x.time()))
truth_df["track_id_true"] = truth_df["track_id"]

output = run_model(truth_df, model)
output.to_csv("ml_out_ds1.csv", index=False)

100%|██████████| 102861/102861 [10:55<00:00, 156.98it/s]


In [35]:
truth_df = pd.read_csv(data_loc("dataset2_truth.csv"))

truth_df["time"] = pd.to_datetime(truth_df["time"])
truth_df["time"] = truth_df["time"].apply(lambda x: datetime.combine(datetime(1970, 1, 1, 0, 0, 0).date(), x.time()))
truth_df["track_id_true"] = truth_df["track_id"]

output = run_model(truth_df, model)
output.to_csv("ml_out_ds2.csv", index=False)

100%|██████████| 97592/97592 [08:09<00:00, 199.20it/s]


In [36]:
truth_df = pd.read_csv(data_loc("dataset3_truth.csv"))

truth_df["time"] = pd.to_datetime(truth_df["time"])
truth_df["time"] = truth_df["time"].apply(lambda x: datetime.combine(datetime(1970, 1, 1, 0, 0, 0).date(), x.time()))
truth_df["track_id_true"] = truth_df["track_id"]

output = run_model(truth_df, model)
output.to_csv("ml_out_ds3.csv", index=False)

100%|██████████| 109666/109666 [11:34<00:00, 157.81it/s]


In [37]:
import atd2025
import pandas as pd
import numpy as np

from ucf_atd_model.data import data_loc
from ucf_atd_model.c20_consts import *

import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import contextily as cx
import pyproj as pp
from datetime import datetime

In [38]:
score1, opacc = atd2025.accuracy.evaluate_predictions("ml_out_ds1.csv", data_loc("dataset1_truth.csv"), return_array=True)
score2, opacc2 = atd2025.accuracy.evaluate_predictions("ml_out_ds2.csv", data_loc("dataset2_truth.csv"), return_array=True)
score3, opacc3 = atd2025.accuracy.evaluate_predictions("ml_out_ds3.csv", data_loc("dataset3_truth.csv"), return_array=True)

In [39]:
print(score1)
print(score2)
print(score3)

0.5275323008720506
0.559128822034593
0.548146189338537
